> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# 图分析（Graph Analytics）

欢迎来到 Semantica 图分析能力的**全面走读**。本 notebook 不止步于简单建图，而是演示一条全生命周期的生产级流水线。

我们模拟一个脏乱的真实场景——**创业生态**（投资人、创业公司、创始人），带你走完流程的每一步：

1. **验证**：坏数据进图之前拦下
2. **清洗**：实体去重、冲突解决
3. **结构分析**：看清网络的形状与健康度
4. **深度分析**：中心性、社区、路径查找
5. **时序分析**：在图数据里时间旅行
6. **溯源**：追踪数据从哪来

### 先看主线

这是 introduction 系列的总装实战篇：单件技能在这里串成一条**脏数据进、可信图出**的流水线。脏数据有四宗罪——重复（"TechFlow AI" vs "TechFlow Inc."）、冲突（同一家公司两种营收）、错误（指向不存在节点的悬空边）、时变（不同时间发生的投资轮）——每个阶段各治一罪。六个阶段三段——**洗 → 用 → 信**：

**进门三道闸（Phase 1–3）—— 洗数据**

1. Phase 1 验证：守门人检查结构完整、schema 合规、ID 唯一——悬空边当场自动修
2. Phase 2 去重与冲突：18 篇的去重 + 17 篇的冲突解决联合执法
3. Phase 3 建图：干净数据组装成正式图对象（07 篇）

**分析两翼（Phase 4–5）—— 用数据**

4. Phase 4 深度分析：连通性、中心性、社区（10 篇的工具 + 10b 篇的指标直觉实战）
5. Phase 5 时序分析：静态图没意思，真实世界会变——按时间点回看图的历史形态

**信任闭环（Phase 6）—— 信数据**

6. Phase 6 溯源：生产系统必须答得出"这条事实从哪来"（22 篇）

开始！

In [ ]:
!pip install -q semantica

In [ ]:
import logging
import json
from datetime import datetime

# Set up logging to see what's happening under the hood
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Import all the powerful tools from Semantica
from semantica.kg import (
    GraphBuilder,
    GraphAnalyzer,
    GraphValidator,
    ConnectivityAnalyzer,
    CentralityCalculator,
    CommunityDetector,
    TemporalGraphQuery,
    ProvenanceTracker
)
from semantica.deduplication import DuplicateDetector
from semantica.conflicts import ConflictDetector, ConflictResolver

## 1. 场景：一个脏乱的创业生态

数据来自多个来源（爬虫、新闻、用户提交）。它很脏：
- **重复**："TechFlow AI" 和 "TechFlow Inc."
- **冲突**：同一家公司两套营收数字
- **错误**：指向不存在节点的关系（悬空边）
- **历史**：不同时间发生的投资轮

In [ ]:
# Our "Raw" Messy Data
raw_entities = [
    {"id": "startup_1", "type": "Startup", "name": "TechFlow AI", "revenue": 1000000, "founded": "2021-01-01"},
    {"id": "startup_2", "type": "Startup", "name": "GreenEnergy Co", "revenue": 500000, "founded": "2020-05-15"},
    {"id": "startup_1_dup", "type": "Startup", "name": "TechFlow Inc.", "revenue": 1200000, "founded": "2021-01-01"}, # Duplicate!
    {"id": "investor_1", "type": "Investor", "name": "Venture Capital X"},
    {"id": "founder_1", "type": "Person", "name": "Alice Chen"},
    {"id": "founder_2", "type": "Person", "name": "Bob Smith"}
]

raw_relationships = [
    # Valid Relationships
    {"source": "founder_1", "target": "startup_1", "type": "FOUNDED", "valid_from": "2021-01-01"},
    {"source": "investor_1", "target": "startup_1", "type": "INVESTED_IN", "amount": 5000000, "valid_from": "2023-06-01"},
    
    # Dangling Edge (Error!)
    {"source": "founder_2", "target": "startup_999", "type": "FOUNDED", "valid_from": "2020-05-15"}, 
    
    # Temporal Data (History)
    {"source": "founder_1", "target": "startup_2", "type": "ADVISED", "valid_from": "2020-01-01", "valid_until": "2021-01-01"}
]

print(f"Loaded {len(raw_entities)} raw entities and {len(raw_relationships)} raw relationships.")

## 2. Phase 1：验证（守门人）

动手之前先验证。坏数据进 = 坏洞察出。`GraphValidator` 检查：
- **结构完整性**：关系的两端都在吗？
- **schema 合规**：实体有必填字段吗？
- **一致性**：ID 唯一吗？

In [ ]:
# Initialize Validator
validator = GraphValidator()

# Create a temporary graph object for validation
temp_graph = {"entities": raw_entities, "relationships": raw_relationships}

# Run Validation
print("Running Validation Check...")
validation_result = validator.validate(temp_graph)

if not validation_result.is_valid:
    print("Validation Failed! Issues found:")
    for issue in validation_result.issues:
        print(f"   - [{issue.severity.name}] {issue.message} (Code: {issue.code})")
        
        # AUTOMATIC FIX: If it's a dangling edge, remove it
        if issue.code == "DANGLING_EDGE":
            print("     Auto-Fixing: Removing invalid relationship...")
            raw_relationships = [r for r in raw_relationships 
                               if r['target'] != issue.details.get('target_id')]
else:
    print("Graph is valid!")

# Re-validate to confirm fix
print("\nRe-validating after fixes...")
temp_graph = {"entities": raw_entities, "relationships": raw_relationships}
if validator.validate(temp_graph).is_valid:
    print("Graph is now clean and valid!")

## 3. Phase 2：去重与冲突解决

"TechFlow AI" 和 "TechFlow Inc." 大概率是同一家公司。营收数据还打架。去重与冲突解决联合执法——18 篇和 17 篇的技能在这里落地。

In [ ]:
# 1. Detect Duplicates
print("Scanning for duplicates...")
deduper = DuplicateDetector(similarity_threshold=0.7) # 70% similarity threshold
duplicates = deduper.detect_duplicates(raw_entities)

for candidate in duplicates:
    print(f"Found potential duplicate pair (Score: {candidate.similarity_score:.2f}):")
    print(f"   - {candidate.entity1['name']} (ID: {candidate.entity1['id']})")
    print(f"   - {candidate.entity2['name']} (ID: {candidate.entity2['id']})")
    
    # MERGE STRATEGY: Keep entity1, merge data from entity2
    print("   Merging entities...")
    # (In a real app, you'd use EntityMerger, but here's the logic:)
    # We keep startup_1 and discard startup_1_dup, but we note the conflict
    
# 2. Detect Conflicts
print("\nChecking for data conflicts...")
conflict_detector = ConflictDetector()

# Simulating a conflict check between the two versions of TechFlow
# To check conflicts, we treat them as the same entity (same ID)
entity_a = raw_entities[0].copy()
entity_b = raw_entities[2].copy()
entity_b['id'] = entity_a['id'] # Force same ID for conflict detection

conflicts = conflict_detector.detect_conflicts([entity_a, entity_b])

for conflict in conflicts:
    print(f"   Conflict detected in field '{conflict.property_name}':")
    print(f"      Values: {conflict.conflicting_values}")
    
    # RESOLUTION: Trust the higher number (optimistic!)
    if conflict.property_name == "revenue":
        # values are strings or ints, need to handle types
        vals = [float(v) for v in conflict.conflicting_values if v is not None]
        resolved_val = max(vals)
        print(f"      Resolved to: {resolved_val}")
        raw_entities[0]['revenue'] = resolved_val

# Final Cleanup: Remove the duplicate entity from our list
clean_entities = [e for e in raw_entities if e['id'] != 'startup_1_dup']
clean_relationships = raw_relationships # (We'd normally re-link relationships too)

print(f"\nCleaned Data: {len(clean_entities)} entities remaining.")

## 4. Phase 3：构建知识图谱

数据干净了，组装正式的图对象。

In [ ]:
# Manual Graph Construction (since we already cleaned it)
kg = {
    "entities": clean_entities,
    "relationships": clean_relationships,
    "metadata": {
        "created_at": datetime.now().isoformat(),
        "source": "Manual Advanced Pipeline"
    }
}
print("Knowledge Graph Assembled Successfully!")

## 5. Phase 4：高级分析

见证奇迹的时刻。多个分析器轮番上——10 篇的工具、10b 篇的指标直觉，在这里对真实数据开火。

In [ ]:
# Initialize the Master Analyzer
analyzer = GraphAnalyzer(enable_temporal=True)

# 1. Structural Analysis (Connectivity)
print("\n--- Connectivity Analysis ---")
connectivity = analyzer.analyze_connectivity(kg)
print(f"   • Graph Connected? {'Yes' if connectivity['is_connected'] else 'No'}")
print(f"   • Connected Components: {connectivity['num_components']}")

# 2. Centrality (Who is important?)
print("\n--- Centrality Analysis ---")
centrality_result = analyzer.calculate_centrality(kg, centrality_type="degree")
degree_data = centrality_result["centrality_measures"]["degree"]

# Get pre-calculated rankings
top_nodes = degree_data["rankings"][:3]

print("   • Top Influencers (Degree Centrality):")
for item in top_nodes:
    print(f"     - {item['node']}: {item['score']:.2f}")

# 3. Community Detection (Clustering)
print("\n--- Community Detection ---")
community_result = analyzer.detect_communities(kg, algorithm="louvain")
communities = community_result["communities"]

print(f"   • Detected {len(communities)} communities.")
for i, comm in enumerate(communities):
    # comm is a set of node IDs
    members = list(comm)
    print(f"     Community {i+1}: {', '.join(members)}")

## 6. Phase 5：时序分析（时间旅行）

静态图很无聊，真实世界会变。来分析生态系统的**演化**——同一个查询，站在 2020 年和站在 2023 年，看到的是不同的图。

In [ ]:
temporal_engine = TemporalGraphQuery(temporal_granularity="year")

# 1. Time Travel Query: What did the world look like in 2020?
print("\n--- Time Travel: 2020 ---")
snapshot_2020 = temporal_engine.query_at_time(kg, query="*", at_time="2020-06-01")
print(f"   Active Relationships in 2020: {len(snapshot_2020['relationships'])}")
for rel in snapshot_2020['relationships']:
    print(f"   - {rel['source']} --[{rel['type']}]--> {rel['target']}")

# 2. Time Travel Query: What about 2023?
print("\n--- Time Travel: 2023 ---")
snapshot_2023 = temporal_engine.query_at_time(kg, query="*", at_time="2023-07-01")
print(f"   Active Relationships in 2023: {len(snapshot_2023['relationships'])}")
for rel in snapshot_2023['relationships']:
    print(f"   - {rel['source']} --[{rel['type']}]--> {rel['target']}")
    
# Notice how 'ADVISED' might disappear if it ended, and 'INVESTED_IN' appears!

## 7. Phase 6：溯源（数据血缘）

最后，生产系统必须知道一条事实**从哪来**——信任的根基。

In [ ]:
tracker = ProvenanceTracker()

# Let's pretend we're tracking the source of our data
tracker.track_entity("startup_1", source="Crunchbase_API_v2", metadata={"confidence": 0.95})
tracker.track_entity("startup_1", source="Manual_Entry_User_Bob", metadata={"confidence": 1.0})

print("\n--- Provenance Report: TechFlow AI ---")
lineage = tracker.get_lineage("startup_1")
print(f"   Entity: startup_1")
print(f"   First Seen: {lineage['first_seen']}")
print(f"   Sources:")
for src in lineage['sources']:
    print(f"     - {src['source']} (at {src['timestamp']})")

## 结语

### 回顾主线

**洗 → 用 → 信** 全程走完：三道闸（验证、去重冲突、建图）把脏数据洗净；两翼（深度分析、时序旅行）把价值榨出；溯源闭环把信任补齐。introduction 的单件技能，串起来就是一条现代知识图谱工程的生产流水线。

你刚刚走完一条完整的高级知识图谱流水线：

1. **验证**了脏乱的输入数据
2. **清洗**了重复与冲突
3. **分析**了结构与社区动态
4. **查询**了时间维度
5. **追踪**了数据血缘

这就是用 Semantica 做现代知识图谱工程的状态-of-the-art。